In [ ]:
import streamlit as st
import pandas as pd
import pickle

try:
    with open('best_model.pkl', 'rb') as f:
        model = pickle.load(f)
    with open('preprocessor.pkl', 'rb') as f:
        preprocessor = pickle.load(f)
except FileNotFoundError:
    st.error("⚠️ Model files not found! Make sure 'best_model.pkl' and 'preprocessor.pkl' are in the same folder.")

st.title("💰 YouTube Revenue Predictor")

# --- 2. USER INTERFACE ---
col1, col2 = st.columns(2)

with col1:
    views = st.number_input("Total Views", min_value=0, value=1000)
    likes = st.number_input("Likes", min_value=0, value=50)
    watch_time = st.number_input("Watch Time (Minutes)", min_value=0.0, value=100.0)

with col2:
    category = st.selectbox("Category", ['Tech', 'Gaming', 'Entertainment', 'Lifestyle', 'Music'])
    country = st.selectbox("Country", ['US', 'IN', 'UK', 'CA', 'AU'])
    device = st.radio("Device Type", ['Mobile', 'Desktop', 'Tablet'])

# --- 3. PREDICTION LOGIC ---
if st.button("Calculate Estimated Revenue"):
    # Create input DataFrame with ALL features used during training
    # Ensure column names match your training data EXACTLY
    user_data = pd.DataFrame({
        'views': [views],
        'watch_time_minutes': [watch_time],
        'likes': [likes],
        'category': [category],
        'comments': [0],
        'video_length_minutes': [10.0],
        'subscribers': [1000],
        'engagement_rate': [0.05],
        'avg_view_duration': [2.0],
        'retention_rate': [0.5],
        'month': [3],
        'is_weekend': [0],
        'device': [device],
        'country': [country]
    })

    # STEP A: Transform the data using your saved preprocessor
    processed_data = preprocessor.transform(user_data)

    # STEP B: Predict using the transformed data
    prediction = model.predict(processed_data)

    st.success(f"Estimated Ad Revenue: ${prediction[0]:,.2f}")

In [1]:
import pickle
import pandas as pd
import numpy as np

# 1. Load the saved tools
with open('preprocessor.pkl', 'rb') as f:
    loaded_preprocessor = pickle.load(f)

with open('best_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

# 2. Create the Raw Input Data (Match your original CSV columns)
test_data = pd.DataFrame({
    'views': [10000],
    'likes': [800],
    'comments': [150],
    'watch_time_minutes': [45000.0],
    'video_length_minutes': [12.0],
    'subscribers': [5000],
    'category': ['Tech'],
    'device': ['Mobile'],
    'country': ['US'],
    'month': [6],
    'is_weekend': [0]
})

# 3. MANUALLY ADD the Feature Engineering columns (This fixes the ValueError!)
test_data['engagement_rate'] = (test_data['likes'] + test_data['comments']) / (test_data['views'] + 1)
test_data['avg_view_duration'] = test_data['watch_time_minutes'] / (test_data['views'] + 1)
test_data['retention_rate'] = test_data['watch_time_minutes'] / (test_data['views'] * test_data['video_length_minutes'] + 1)
test_data['view_to_sub_ratio'] = test_data['views'] / (test_data['subscribers'] + 1)
test_data['comment_to_like_ratio'] = test_data['comments'] / (test_data['likes'] + 1)

# 4. ADD the Interaction Features (The ones that caused the error)
test_data['performance_score'] = (test_data['views'] * 0.5 + test_data['likes'] * 0.3 + test_data['comments'] * 0.2)
test_data['total_attention'] = test_data['views'] * test_data['watch_time_minutes']
test_data['quality_engagement'] = test_data['likes'] * test_data['watch_time_minutes']

# 5. Transform and Predict
try:
    X_new_pre = loaded_preprocessor.transform(test_data)
    prediction = loaded_model.predict(X_new_pre)
    print(f"✅ Success! Predicted Revenue: ${prediction[0]:.2f}")
except Exception as e:
    print(f"❌ Still missing something: {e}")

✅ Success! Predicted Revenue: $280.48
